In [1]:
from pathlib import Path

from einops import rearrange
import matplotlib as mpl
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import torch
from torch import Tensor

from train import Trainer
from src.interfaces import DatasetBase
from src.datasets.addition import AdditionDataset
from src.datasets.subtraction import SubtractionDataset

mpl.rcParams['figure.dpi'] = 300

### Utils

In [2]:
def pretty_print_sample(dataset: DatasetBase, x: Tensor, verbose: bool = True) -> str:
    steps = [
        dataset.to_string(x[i]).split("\n")
        for i in range(x.shape[0])
    ]

    x_str = "\n".join(
        "   ".join(
            steps[i][j]
            for i in range(len(steps))
        )
        for j in range(len(steps[0]))
    )

    if verbose:
        print(x_str)

    return x_str

### Visualize Specific Dataset

In [12]:
dataset = SubtractionDataset(seed=31)
sample = dataset.get_example()
_ = pretty_print_sample(dataset,sample)

  ____     ____     __1_     __1_     _11_     _11_     011_     011_
   826      826      826      826      826      826      826      826
-  667   -  667   -  667   -  667   -  667   -  667   -  667   -  667
______   ______   ______   ______   ______   ______   ______   ______
  ____     ___9     ___9     __59     __59     _159     _159     0159
                     


### Checkpoint options

In [10]:
# If you change this, revert before committing
name = "enc_tokens"
day = "2025-11-18"
time = "14-04-02"
ckpt = "ckpt_20.pth"
output_dir = Path.cwd().parent / "outputs" / name / day / time

config_file = output_dir / ".hydra" / "config.yaml"
cfg = OmegaConf.load(config_file)

cfg.resume_ckpt = output_dir / "checkpoints" / ckpt
# cfg.dataset.max_digits = 4

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/g/Projects/Deep-Learning-2025/outputs/enc_tokens/2025-11-18/14-04-02/.hydra/config.yaml'

### Load Model

In [ ]:
trainer = Trainer(cfg, output_dir)
trainer.model.eval()

NameError: name 'cfg' is not defined

### Analysis

#### Get Sample

In [ ]:
dataset = trainer.dataset

In [ ]:
sample = trainer.fabric.to_device(dataset.get_example())
sample_str = pretty_print_sample(dataset, sample)

#### Run Model

In [ ]:
x0 = sample[0:1]
x0_str = dataset.to_string(x0[0])

print(x0.shape)
print(x0_str)

In [ ]:
x0 = sample[0:1]
x0_str = dataset.to_string(x0[0])

steps = [x0.clone()]
enc_attnss = []
y_preds = []

for _ in range(sample.shape[0]-1):
  with torch.no_grad():
      _, _, y_pred, _, _, enc_attns, _, _ = trainer.forward((x0, None))
      out = trainer.head.step(x0, y_pred)

      steps.append(out)
      y_preds.append(y_pred)
      enc_attnss.append(enc_attns)
      x0 = out

out_str = pretty_print_sample(dataset, torch.cat(steps, dim=0), verbose=False)

print(f"=== Solution ===\n{sample_str}")
print(f"=== Prediction ===\n{out_str}")

#### Plot Probabilities

In [ ]:
y_pred.shape

In [ ]:
loss, acc = trainer.head.forward_loss(torch.randn_like(y_pred), steps[-1], steps[-1])
loss.item()

In [ ]:
[y.shape for y in steps]

In [ ]:
b, h, w = steps[0].shape
y_preds = list(map(lambda x: rearrange(x, "b (h w) v -> b h w v", h=h, w=w), y_preds))

In [ ]:
step = 0
x0_str = dataset.to_string(steps[step][0])
out_str = dataset.to_string(steps[step+1][0])
enc_attns = enc_attnss[step]
y_pred = y_preds[step]

_, h, w, n = y_pred.shape

y_pred_flat = rearrange(y_pred, "b h w v -> b (h w v)")
probs = torch.softmax(y_pred_flat, dim=1)
probs = rearrange(probs, "b (h w v) -> b h w v", h=h, w=w, v=n)
probs = probs[..., :-1, :]  # ignore \n

# token = "9"  # token to plot
for token in dataset.token_to_idx:
  ti = dataset.token_to_idx.index(token)
  plt.imshow(probs[0, :, :, ti].cpu(), cmap='magma', vmin=0.0, vmax=1.0)
  for y in range(h):
    for x in range(w - 1):
        text = x0_str.split("\n")[y][x]
        plt.text(x, y, text, ha='center', va='center', color='white' if probs[0, y, x, ti] < 0.5 else "black", fontsize=8)
  plt.title(f"Probs for token {token} ({100*probs[0, :, :, ti].max():.2f}%)")
  plt.colorbar()
  plt.show()

#### Plot Attention Maps

In [ ]:
n_layers, _, n_heads, s, _ = enc_attns.shape # (n_layers, b, h, s, s)
n_registers = trainer.head.n_registers
h, w = dataset.h, dataset.w
n_layers, n_heads, s, n_registers, h, w

##### Between Input Tokens

In [ ]:
i, j = 4, 5
fig, axes = plt.subplots(n_layers, n_heads, figsize=(12, 8))
for layer in range(n_layers):
    for head in range(n_heads):
      ax = axes[layer, head]
      attn_map = enc_attns[layer, 0, head, n_registers:, n_registers:]
      attn_map = rearrange(attn_map, "(ha wa) (hb wb) -> ha wa hb wb", ha=h, hb=h, wa=w, wb=w)
      im = ax.imshow(attn_map[i, j, :, :-1].cpu(), aspect='auto', cmap='magma')

      vmin, vmax = attn_map[i, j, :, :-1].min().item(), attn_map[i, j, :, :-1].max().item()
      vhalf = (vmin + vmax) / 2

      for y in range(h):
        for x in range(w - 1):
            text = "X" if (y, x) == (i, j) else x0_str.split("\n")[y][x]
            ax.text(x, y, text, ha='center', va='center', color='white' if attn_map[i, j, y, x] < vhalf else "black", fontsize=8)
      ax.set_title(f'Layer {layer+1}, Head {head+1}')
      fig.colorbar(im, ax=ax)

plt.suptitle(f'Attention Maps for token at position ({i}, {j})')
plt.tight_layout()
plt.show()

##### Between Latents and Input Tokens

In [ ]:
for register in range(n_registers):
    fig, axes = plt.subplots(n_layers, n_heads, figsize=(12, 8))
    for layer in range(n_layers):
        for head in range(n_heads):
          ax = axes[layer, head]
          attn_map = enc_attns[layer, 0, head, register, n_registers:]
          attn_map = rearrange(attn_map, "(ha wa) -> ha wa", ha=h, wa=w)

          vmin, vmax = attn_map.min().item(), attn_map.max().item()
          vhalf = (vmin + vmax) / 2

          im = ax.imshow(attn_map[:, :-1].cpu(), aspect='auto', cmap='magma')
          for y in range(h):
            for x in range(w - 1):
                text = x0_str.split("\n")[y][x]
                ax.text(x, y, text, ha='center', va='center', color='white' if attn_map[y, x] < vhalf else "black", fontsize=8)
          ax.set_title(f'Layer {layer+1}, Head {head+1}')
          fig.colorbar(im, ax=ax)

    plt.suptitle(f'Attention Maps from Register {register} to Input Tokens')
    plt.tight_layout()
    plt.show()